In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ["MUJOCO_GL"] = "egl"

In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import argparse
import pathlib
import sys

import numpy as np
import torch
#from omegaconf import OmegaConf

import sys

sys.path.append("/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch")
import dreamer as dreamer_main
import models as dreamer_models
import tools as dreamer_tools
import ruamel.yaml as yaml

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:
import pickle

with open(
        "/home/hgf_hmgu/hgf_gib4562/tdmpc2/analysis/data_generation/data_off_policy.pkl",
        "rb") as f:
    data_off_policy = pickle.load(f)

In [5]:
n = len(data_off_policy['obs_states'])
obs_batch = {'state': {}, 'pixel': {}}

# State features
obs_batch['state']['position'] = data_off_policy['obs_states'][:, :3]
obs_batch['state']['velocity'] = data_off_policy['obs_states'][:, 3:]

obs_batch['pixel']['image'] = data_off_policy['obs_pixels']
obs_batch['state']['action'] = np.concatenate(data_off_policy['actions'],
                                              axis=0)
obs_batch['pixel']['action'] = np.concatenate(data_off_policy['actions'],
                                              axis=0)
obs_batch['state']['is_first'] = np.zeros((n, 1), dtype=np.float32)
obs_batch['state']['is_first'][::500] = 1
obs_batch['pixel']['is_first'] = np.zeros((n, 1), dtype=np.float32)
obs_batch['pixel']['is_first'][::500] = 1
obs_batch['state']['is_terminal'] = np.zeros((n, 1), dtype=np.float32)
obs_batch['pixel']['is_terminal'] = np.zeros((n, 1), dtype=np.float32)

for k in obs_batch['state']:
    obs_batch['state'][k] = np.expand_dims(obs_batch['state'][k], axis=0)

for k in obs_batch['pixel']:
    obs_batch['pixel'][k] = np.expand_dims(obs_batch['pixel'][k], axis=0)

# obs_batch['pixel']['image'] = np.expand_dims(obs_batch['pixel']['image'],
#                                              axis=0)

In [6]:
obs_batch['state']['position'].shape, obs_batch['state'][
    'velocity'].shape, obs_batch['state']['action'].shape, obs_batch['pixel'][
        'image'].shape

((1, 25000, 3), (1, 25000, 2), (1, 25000, 1), (1, 25000, 64, 64, 3))

In [7]:
obs_batch['state']['position'].ndim

3

In [8]:
# Real Dreamer obs_batch from DMC cartpole_swingup (vision + proprio)
import numpy as np
import torch

from envs import dmc
import envs.wrappers as wrappers

# T = 100  # rollout length

# Create env (matches Dreamer DMC defaults)
env = dmc.DeepMindControl("cartpole_swingup",
                          action_repeat=2,
                          size=(64, 64),
                          seed=0)
env = wrappers.NormalizeActions(env)


/home/hgf_hmgu/hgf_gib4562/miniconda3/envs/dreamerv3/lib/python3.11/site-packages/gym/spaces/box.py:78: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [9]:
def _parse_dreamer_config(configs_path, config_names, overrides):
    configs = yaml.safe_load(pathlib.Path(configs_path).read_text())

    def recursive_update(base, update):
        for key, value in update.items():
            if isinstance(value, dict) and key in base:
                recursive_update(base[key], value)
            else:
                base[key] = value

    name_list = ["defaults", *config_names] if config_names else ["defaults"]
    defaults = {}
    for name in name_list:
        recursive_update(defaults, configs[name])

    overrides_map = {}
    for item in overrides:
        if item.startswith("--"):
            item = item[2:]
        if "=" in item:
            key, value = item.split("=", 1)
            if key not in defaults:
                raise KeyError(f"Unknown config key '{key}' in override.")
            cast = dreamer_tools.args_type(defaults[key])
            overrides_map[key] = cast(value)
        else:
            if item not in defaults:
                raise KeyError(f"Unknown config key '{item}' in override.")
            overrides_map[item] = True

    merged = {**defaults, **overrides_map}
    return argparse.Namespace(**merged)

In [10]:
def load_dreamer_world_model(
        configs_path="/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/configs.yaml",
        config_names=["dmc_proprio"],
        overrides=["task=dmc_cartpole_swingup"],
        ckpt_path='/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/logdir/dmc_cartpole_swingup_srun/latest.pt',
        env=None,
        device='cuda:0'):
    cfg = _parse_dreamer_config(
        configs_path,
        config_names,
        overrides,
    )
    cfg.num_actions = 1
    assert env is not None, "You must provide an environment with a valid observation_space."
    wm = dreamer_models.WorldModel(env.observation_space,
                                   None,
                                   step=0,
                                   config=cfg).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    agent_state = ckpt.get("agent_state_dict", ckpt)

    # Handle torch.compile() prefix - strip '_orig_mod.' from keys
    agent_state = {
        k.replace('_orig_mod.', ''): v for k, v in agent_state.items()
    }

    wm_state = {
        k[len("_wm."):]: v
        for k, v in agent_state.items()
        if k.startswith("_wm.")
    }
    wm.load_state_dict(wm_state, strict=True)
    wm.eval()
    return wm, cfg

In [11]:
# Example usage (you must provide 'env'):
wm_state, cfg_state = load_dreamer_world_model(
    configs_path=
    "/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/configs.yaml",
    config_names=["dmc_proprio"],
    overrides=["task=dmc_cartpole_swingup"],
    ckpt_path=
    '/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/logdir/dmc_cartpole_swingup_srun/latest.pt',
    env=env)

Encoder CNN shapes: {}
Encoder MLP shapes: {'position': (3,), 'velocity': (2,)}
Decoder CNN shapes: {}
Decoder MLP shapes: {'position': (3,), 'velocity': (2,)}


/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/tools.py:747: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/scratch/slurm_tmpdir/job_1610219/ipykernel_2979490/3701743254.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend y

Optimizer model_opt has 16428293 variables.


In [12]:
# Example usage (you must provide 'env'):
wm_pixel, cfg_pixel = load_dreamer_world_model(
    configs_path=
    "/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/configs.yaml",
    config_names=["dmc_vision"],
    overrides=["task=dmc_cartpole_swingup"],
    ckpt_path=
    '/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/logdir/dmc_cartpole_swingup_pixel/latest.pt',
    env=env)

Encoder CNN shapes: {'image': (64, 64, 3)}
Encoder MLP shapes: {}
Decoder CNN shapes: {'image': (64, 64, 3)}
Decoder MLP shapes: {}
Optimizer model_opt has 15685251 variables.


/scratch/slurm_tmpdir/job_1610219/ipykernel_2979490/3701743254.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


In [13]:
def get_feats_all(wm, obs_batch):
    data = wm.preprocess(obs_batch)
    embed = wm.encoder(data)
    post, _ = wm.dynamics.observe(embed, data["action"], data["is_first"])
    feats = wm.dynamics.get_feat(post)  # (B, T, feat_dim)

    probs = torch.softmax(post["logit"], dim=-1)  # (B,T,stoch,discrete)
    state_soft = {**post, "stoch": probs}
    feat_soft = wm.dynamics.get_feat(state_soft)
    return feats, feat_soft


In [14]:
def get_feats_all_in_batches(wm, obs_batch, batch_size=256):
    """
    Compute features for all observations in obs_batch, processing in batches.

    Args:
        wm: the world model
        obs_batch: dict of numpy arrays/tensors, each (N, T, ...)
        batch_size: int, how many sequences to process at once

    Returns:
        feats_all: (N, T, feat_dim)
        feat_soft_all: (N, T, feat_dim)
    """
    import torch

    # Ensure obs_batch is a dict with arrays of same length in 0th dim
    total_n = 25_000

    # Prepare list of batched outputs
    feats_list = []
    feat_soft_list = []

    from tqdm import tqdm
    for start in tqdm(range(0, total_n, batch_size)):
        end = min(start + batch_size, total_n)

        # Prepare mini-batch dict
        mini_batch = {}
        for k, v in obs_batch.items():
            if isinstance(v, torch.Tensor):
                mini_batch[k] = v[:, start:end]
            else:
                # assume ndarray, convert to torch
                mini_batch[k] = torch.from_numpy(v[:, start:end]).to(
                    next(wm.parameters()).device)
        # print(mini_batch['image'].shape, mini_batch['action'].shape,
        #       mini_batch['is_first'].shape)
        data = wm.preprocess(mini_batch)
        embed = wm.encoder(data)
        post, _ = wm.dynamics.observe(embed, data["action"], data["is_first"])
        feats = wm.dynamics.get_feat(post)  # (B, T, feat_dim)

        probs = torch.softmax(post["logit"], dim=-1)  # (B,T,stoch,discrete)
        state_soft = {**post, "stoch": probs}
        feat_soft = wm.dynamics.get_feat(state_soft)

        feats_list.append(feats.squeeze())
        feat_soft_list.append(feat_soft.squeeze())

    feats_all = torch.cat(feats_list, dim=0)
    feat_soft_all = torch.cat(feat_soft_list, dim=0)
    return feats_all, feat_soft_all


In [15]:
feats_onehot_pixel, feat_soft_pixel = get_feats_all_in_batches(
    wm_pixel, obs_batch['pixel'], batch_size=32)
feats_onehot_pixel = feats_onehot_pixel.squeeze().detach().cpu().numpy()
feat_soft_pixel = feat_soft_pixel.squeeze().detach().cpu().numpy()

  0%|          | 0/782 [00:00<?, ?it/s]

/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/models.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  k: torch.tensor(v, device=self._config.device, dtype=torch.float32)
100%|██████████| 782/782 [01:03<00:00, 12.40it/s]


In [16]:
#Example usage:
feats_onehot_state, feat_soft_state = get_feats_all_in_batches(
    wm_state, obs_batch['state'])
feats_onehot_state = feats_onehot_state.squeeze().detach().cpu().numpy()
feat_soft_state = feat_soft_state.squeeze().detach().cpu().numpy()


  0%|          | 0/98 [00:00<?, ?it/s]/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/models.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  k: torch.tensor(v, device=self._config.device, dtype=torch.float32)
100%|██████████| 98/98 [00:59<00:00,  1.64it/s]


In [17]:
feats_onehot_state.shape, feats_onehot_pixel.shape, feat_soft_state.shape, feat_soft_pixel.shape

((25000, 1536), (25000, 1536), (25000, 1536), (25000, 1536))

In [18]:
results_dreamer = {}

results_dreamer['z_dreamer_onehot_state'] = feats_onehot_state
results_dreamer['z_dreamer_onehot_pixel'] = feats_onehot_pixel
results_dreamer['z_dreamer_soft_state'] = feat_soft_state
results_dreamer['z_dreamer_soft_pixel'] = feat_soft_pixel

In [19]:
import pickle

In [20]:
with open(
        "/home/hgf_hmgu/hgf_gib4562/tdmpc2/analysis/outputs_tdmpc2_vs_dreamer/dreamer_features_pixel_vs_state.pkl",
        "wb") as f:
    pickle.dump(results_dreamer, f)